# V14: 팀원 공유 모델의 인사이트와 v13의 앙상블 구조 통합

이 노트북은 v13의 앙상블(XGB+LGBM) 구조에 팀원이 제안한 참여율 기반 예측 및 세부 피처 엔지니어링, 그리고 날씨/공휴일/메뉴 보정 개념을 통합한 버전입니다.

## 주요 전략
1. **참여율 기반 예측**: 전체 인원수가 아닌 참여율(참여인원 / 식사가능인원)을 타겟으로 학습
2. **피처 엔지니어링 강화**: 
    - 팀원 모델의 `days_to_month_end`, `is_month_end_part`, 요일 특성(수/금) 반영
    - 메뉴 키워드 세분화 및 석식 스타일링 반영
    - 날씨 보정 신호(is_rain, is_hot, is_cold)를 피처에 포함
3. **앙상블 고도화**: XGBoost + CatBoost + LightGBM (1:1:1 결합)
4. **교차 검증**: 5-Fold KFold를 통한 안정적 학습

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from korean_font import set_korean_font
import warnings

set_korean_font()
warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)

한글 폰트 설정: Malgun Gothic (c:/Windows/Fonts/malgun.ttf)


In [2]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
weather_train = pd.read_csv('data/weather.csv')
weather_test = pd.read_csv('data/weatherTest.csv')

# 날씨 데이터 통합 및 전처리
weather = pd.concat([weather_train, weather_test], axis=0)
weather['일시'] = pd.to_datetime(weather['일시'])
weather['기온'] = pd.to_numeric(weather['기온'], errors='coerce').interpolate().bfill().ffill()
weather['강수량'] = pd.to_numeric(weather['강수량'], errors='coerce').fillna(0)
weather['is_rain'] = (weather['강수량'] > 0).astype(int)
weather['is_hot'] = (weather['기온'] >= 28).astype(int)
weather['is_cold'] = (weather['기온'] <= 5).astype(int)
weather = weather[['일시', '기온', '습도', 'is_rain', 'is_hot', 'is_cold']]

In [3]:
# 2. 피처 엔지니어링 함수
def normalize_menu_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('/', ' ').replace('&', ' ').replace('+', ' ')
    text = text.replace('(', ' ').replace(')', ' ')
    text = re.sub(r'[^가-힣A-Za-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

lunch_keyword_map = {
    'chicken': '치킨', 'donkatsu': '돈까스', 'jeyuk': '제육', 'curry': '카레',
    'fried_rice': '볶음밥', 'bibimbap': '비빔밥', 'jjajang': '짜장', 'tangsuyuk': '탕수육'
}

def get_lunch_keywords(text):
    text = normalize_menu_text(text)
    return {f'kw_{k}': int(v in text) for k, v in lunch_keyword_map.items()}

def get_dinner_style(text):
    text = normalize_menu_text(text)
    flags = {'style_chinese': 0, 'style_western': 0, 'style_japanese': 0, 'style_snack': 0, 'style_fusion': 0}
    chinese_kw = ['짜장', '짬뽕', '탕수', '마파', '깐풍', '유산슬']
    western_kw = ['돈까스', '스테이크', '파스타', '스프', '햄버거', '피자', '리조또']
    japanese_kw = ['초밥', '우동', '가라아게', '돈부리', '소바']
    snack_kw = ['떡볶이', '순대', '튀김', '라볶이', '김밥', '분식', '어묵']
    fusion_kw = ['브리또', '타코', '샐러드', '또띠아', '퓨전']

    if any(k in text for k in chinese_kw): flags['style_chinese'] = 1
    if any(k in text for k in western_kw): flags['style_western'] = 1
    if any(k in text for k in japanese_kw): flags['style_japanese'] = 1
    if any(k in text for k in snack_kw): flags['style_snack'] = 1
    if any(k in text for k in fusion_kw): flags['style_fusion'] = 1
    return flags

def preprocessing(df):
    df['일자'] = pd.to_datetime(df['일자'])
    df = pd.merge(df, weather, left_on='일자', right_on='일시', how='left')
    df['month'] = df['일자'].dt.month
    df['weekday'] = df['일자'].dt.weekday
    df['day'] = df['일자'].dt.day
    
    # 식사가능자수 및 요일 특성 (팀원 모델 반영)
    df['available'] = (df['본사정원수'] - df['본사휴가자수'] - df['본사출장자수'] - df['현본사소속재택근무자수']).clip(lower=1)
    
    month_end = df['일자'] + pd.offsets.MonthEnd(0)
    df['days_to_month_end'] = (month_end - df['일자']).dt.days
    df['is_month_end_part'] = (df['days_to_month_end'] <= 5).astype(int)
    df['is_wed'] = (df['weekday'] == 2).astype(int)
    df['is_fri'] = (df['weekday'] == 4).astype(int)
    df['has_overtime'] = (df['본사시간외근무명령서승인건수'] > 0).astype(int)
    df['log_overtime'] = np.log1p(df['본사시간외근무명령서승인건수'])
    
    # 공휴일 전후 (v13 + 팀원의 stronger concept 가중치는 피처로)
    df = df.sort_values('일자')
    df['holiday_before'] = (df['일자'].shift(-1).diff().dt.days > 1).astype(int)
    df['holiday_after'] = (df['일자'].diff().dt.days > 1).astype(int)
    
    # 메뉴 피처화
    lunch_menu_df = pd.DataFrame([get_lunch_keywords(x) for x in df['중식메뉴'].fillna("")])
    dinner_menu_df = pd.DataFrame([get_dinner_style(x) for x in df['석식메뉴'].fillna("")])
    
    df = pd.concat([df.reset_index(drop=True), lunch_menu_df, dinner_menu_df], axis=1)
    
    return df.fillna(0)

train_df = preprocessing(train)
test_df = preprocessing(test)

# 타겟 설정 (참여율)
train_df['target_l'] = train_df['중식계'] / train_df['available']
train_df['target_d'] = train_df['석식계'] / train_df['available']

In [4]:
# 3. 모델링 및 학습
lunch_base_features = ['month', 'weekday', 'day', '본사출장자수', '본사시간외근무명령서승인건수', 
                       '기온', '습도', 'is_rain', 'is_hot', 'is_cold', 
                       'holiday_before', 'holiday_after', 'days_to_month_end', 'is_month_end_part', 'is_fri']
lunch_menu_features = [f'kw_{k}' for k in lunch_keyword_map.keys()]
lunch_features = lunch_base_features + lunch_menu_features

dinner_base_features = ['month', 'weekday', 'day', '본사출장자수', '본사시간외근무명령서승인건수', 
                        '기온', '습도', 'is_rain', 'is_hot', 'is_cold', 
                        'holiday_before', 'holiday_after', 'is_wed', 'has_overtime', 'log_overtime']
dinner_menu_features = ['style_chinese', 'style_western', 'style_japanese', 'style_snack', 'style_fusion']
dinner_features = dinner_base_features + dinner_menu_features

def run_kfold_triple_ensemble(train_df, features, target_col, test_df):
    X = train_df[features]
    y = train_df[target_col]
    X_test = test_df[features]
    
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    final_preds = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # XGBoost
        m_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=SEED, n_jobs=-1)
        m_xgb.fit(X_tr, y_tr)
        
        # LightGBM
        m_lgbm = LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
        m_lgbm.fit(X_tr, y_tr)
        
        # CatBoost
        m_cat = CatBoostRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=SEED, verbose=0, thread_count=-1)
        m_cat.fit(X_tr, y_tr)
        
        # 앙상블 (1:1:1)
        fold_preds = (m_xgb.predict(X_test) + m_lgbm.predict(X_test) + m_cat.predict(X_test)) / 3
        final_preds += fold_preds / 5
        
    return final_preds

print("Training Lunch Model...")
res_lunch_ratio = run_kfold_triple_ensemble(train_df, lunch_features, 'target_l', test_df)
print("Training Dinner Model...")
res_dinner_ratio = run_kfold_triple_ensemble(train_df, dinner_features, 'target_d', test_df)

# 인원수로 환산
res_lunch = res_lunch_ratio * test_df['available']
res_dinner = res_dinner_ratio * test_df['available']

Training Lunch Model...
Training Dinner Model...


In [5]:
# 4. 결과 저장
sample = pd.read_csv('data/sample_submission.csv')
sample['중식계'] = res_lunch.values
sample['석식계'] = res_dinner.values

os.makedirs('submission', exist_ok=True)
sample.to_csv('submission/submission_v14.csv', index=False, encoding='utf-8-sig')
print("v14 저장 완료: submission/submission_v14.csv")

v14 저장 완료: submission/submission_v14.csv
